<a href="https://colab.research.google.com/github/ARNAVKS/Named-Entity-Recognition/blob/main/POS_improved.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# POS Tagging — Improved Notebook
**Changes from original:**
- Proper train/val/test split (80/10/10) instead of just validation_split
- Token-level accuracy metric masked for padding (excludes 0s from metric)
- seqeval per-tag F1 on held-out test set
- Baseline BiLSTM saved and evaluated
- DistilBERT fine-tune for ablation comparison
- Clean inference function


## 1. Setup

In [ ]:
# Kaggle setup — skip if data already downloaded
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d naseralqaydeh/named-entity-recognition-ner-corpus
from zipfile import ZipFile
with ZipFile('named-entity-recognition-ner-corpus.zip','r') as f:
    f.extractall()

Dataset URL: https://www.kaggle.com/datasets/naseralqaydeh/named-entity-recognition-ner-corpus
License(s): DbCL-1.0
100% 4.14M/4.14M [00:00<00:00, 18.6MB/s]



In [ ]:
!pip install transformers datasets evaluate seqeval -q

import ast
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from seqeval.metrics import classification_report, f1_score
print('TF version:', tf.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00
TF version: 2.20.0


## 2. Load & Parse Data

In [ ]:
table = pd.read_csv('ner.csv')
table['POS'] = table['POS'].apply(ast.literal_eval)

# words as list — consistent with POS list length
table['words'] = table['Sentence'].apply(lambda x: x.split())

# Sanity check: word count must match POS tag count
mismatches = [(i, len(table['words'][i]), len(table['POS'][i]))
              for i in range(len(table))
              if len(table['words'][i]) != len(table['POS'][i])]
print(f'Mismatches: {len(mismatches)}')  # should be 0
print(f'Total sentences: {len(table)}')
print(table[['words','POS']].head(2))

Mismatches: 4
Total sentences: 47959
                                               words  \
0  [Thousands, of, demonstrators, have, marched, ...   
1  [Families, of, soldiers, killed, in, the, conf...   

                                                 POS  
0  [NNS, IN, NNS, VBP, VBN, IN, NNP, TO, VB, DT, ...  
1  [NNS, IN, NNS, VBN, IN, DT, NN, VBD, DT, NNS, ...  


In [ ]:
table.head()

,Sentence #,Sentence,POS,Tag,words
0,Sentence: 1,Thousands of demonstrators have marched throug...,"[NNS, IN, NNS, VBP, VBN, IN, NNP, TO, VB, DT, ...","['O', 'O', 'O', 'O', 'O', 'O', 'B-geo', 'O', '...","[Thousands, of, demonstrators, have, marched, ..."
1,Sentence: 2,Families of soldiers killed in the conflict jo...,"[NNS, IN, NNS, VBN, IN, DT, NN, VBD, DT, NNS, ...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ...","[Families, of, soldiers, killed, in, the, conf..."
2,Sentence: 3,They marched from the Houses of Parliament to ...,"[PRP, VBD, IN, DT, NNS, IN, NN, TO, DT, NN, IN...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ...","[They, marched, from, the, Houses, of, Parliam..."
3,Sentence: 4,"Police put the number of marchers at 10,000 wh...","[NNS, VBD, DT, NN, IN, NNS, IN, CD, IN, NNS, V...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ...","[Police, put, the, number, of, marchers, at, 1..."
4,Sentence: 5,The protest comes on the eve of the annual con...,"[DT, NN, VBZ, IN, DT, NN, IN, DT, JJ, NN, IN, ...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ...","[The, protest, comes, on, the, eve, of, the, a..."


## 3. Train / Val / Test Split
> **Why this matters:** original notebook used `validation_split=0.2` inside `.fit()` which shuffles differently each run and has no held-out test set. This gives reproducible, comparable numbers.

In [ ]:
train_df, test_df  = train_test_split(table, test_size=0.10, random_state=42)
train_df, val_df   = train_test_split(train_df, test_size=0.10, random_state=42)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

Train: 38846 | Val: 4317 | Test: 4796


## 4. Tokenization (Keras — same as original)

In [ ]:
max_len = max(len(s) for s in table['words'])
print('max_len:', max_len)

sent_tok = Tokenizer(filters='', oov_token='<OOV>')
pos_tok  = Tokenizer(lower=False, filters='')

# Fit only on TRAIN to avoid data leakage
sent_tok.fit_on_texts(train_df['Sentence'])
pos_tok.fit_on_texts(train_df['POS'])

print(f'Vocab size: {len(sent_tok.word_index)} | POS tags: {len(pos_tok.word_index)}')

max_len: 104
Vocab size: 29235 | POS tags: 42


In [ ]:
def encode(df):
    X = pad_sequences(sent_tok.texts_to_sequences(df['Sentence']),
                      maxlen=max_len, padding='post')
    y = pad_sequences(pos_tok.texts_to_sequences(df['POS']),
                      maxlen=max_len, padding='post')
    return X, y

X_train, y_train = encode(train_df)
X_val,   y_val   = encode(val_df)
X_test,  y_test  = encode(test_df)

print('X_train:', X_train.shape, '| y_train:', y_train.shape)

X_train: (38846, 104) | y_train: (38846, 104)


## 5. Baseline: BiLSTM + Softmax (original architecture)
> Keeping this as the **baseline** to compare against DistilBERT. Your original got ~99.4% val accuracy — but that includes padding tokens. Real token accuracy below is lower.

In [ ]:
num_pos_tags = len(pos_tok.word_index) + 1
vocab_size   = len(sent_tok.word_index) + 1

# Masked accuracy — ignores padding (label=0)
class MaskedAccuracy(tf.keras.metrics.Metric):
    def __init__(self, **kwargs):
        super().__init__(name='masked_accuracy', **kwargs)
        self.correct = self.add_weight(name='correct', shape=(), initializer='zeros')
        self.total   = self.add_weight(name='total',   shape=(), initializer='zeros')
    def update_state(self, y_true, y_pred, sample_weight=None):
        y_pred_ids = tf.argmax(y_pred, axis=-1, output_type=tf.int32)
        y_true     = tf.cast(y_true, tf.int32)
        mask       = tf.not_equal(y_true, 0)          # exclude padding
        correct    = tf.equal(y_true, y_pred_ids)
        correct    = tf.logical_and(correct, mask)
        self.correct.assign_add(tf.cast(tf.reduce_sum(tf.cast(correct, tf.int32)), tf.float32))
        self.total.assign_add(tf.cast(tf.reduce_sum(tf.cast(mask, tf.int32)), tf.float32))
    def result(self):
        return self.correct / (self.total + 1e-8)
    def reset_state(self):
        self.correct.assign(0)
        self.total.assign(0)

baseline_model = models.Sequential([
    layers.Embedding(vocab_size, 100, mask_zero=True),
    layers.Bidirectional(layers.LSTM(64, dropout=0.2, return_sequences=True)),
    layers.TimeDistributed(layers.Dense(num_pos_tags, activation='softmax'))
])
baseline_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=[MaskedAccuracy()]
)
baseline_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_2              │ ?                      │   0 (unbuilt) │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

callbacks = [
    EarlyStopping(monitor='val_masked_accuracy', patience=2, restore_best_weights=True,mode = 'max'),
    ModelCheckpoint('pos_baseline.keras', monitor='val_masked_accuracy',
                    save_best_only=True, verbose=1,mode='max')
]

history = baseline_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=32,
    callbacks=callbacks
)

Epoch 1/5
1214/1214 ━━━━━━━━━━━━━━━━━━━━ 0s 293ms/step - loss: 1.2020 - masked_accuracy: 0.6943
Epoch 1: val_masked_accuracy improved from None to 0.96475, saving model to pos_baseline.keras

Epoch 1: finished saving model to pos_baseline.keras
1214/1214 ━━━━━━━━━━━━━━━━━━━━ 380s 303ms/step - loss: 0.5041 - masked_accuracy: 0.8680 - val_loss: 0.1113 - val_masked_accuracy: 0.9648
Epoch 2/5
1214/1214 ━━━━━━━━━━━━━━━━━━━━ 0s 291ms/step - loss: 0.0812 - masked_accuracy: 0.9746
Epoch 2: val_masked_accuracy improved from 0.96475 to 0.97100, saving model to pos_baseline.keras

Epoch 2: finished saving model to pos_baseline.keras
1214/1214 ━━━━━━━━━━━━━━━━━━━━ 377s 300ms/step - loss: 0.0784 - masked_accuracy: 0.9749 - val_loss: 0.0901 - val_masked_accuracy: 0.9710
Epoch 3/5
1214/1214 ━━━━━━━━━━━━━━━━━━━━ 0s 308ms/step - loss: 0.0538 - masked_accuracy: 0.9821
Epoch 3: val_masked_accuracy improved from 0.97100 to 0.97157, saving model to pos_baseline.keras

Epoch 3: finished saving model to pos_

## 6. Evaluate Baseline on Test Set with seqeval

In [ ]:
def get_seqeval_preds(model, X, y_true_ids):
    """Convert model predictions and true labels to seqeval format."""
    idx2pos = pos_tok.index_word   # {1: 'NNS', 2: 'IN', ...}
    preds   = np.argmax(model.predict(X, batch_size=64), axis=-1)

    true_seqs, pred_seqs = [], []
    for true_row, pred_row in zip(y_true_ids, preds):
        tl, pl = [], []
        for t, p in zip(true_row, pred_row):
            if t != 0:                              # skip padding
                tl.append(idx2pos.get(t, 'O'))
                pl.append(idx2pos.get(p, 'O'))
        true_seqs.append(tl)
        pred_seqs.append(pl)
    return true_seqs, pred_seqs

true_seqs, pred_seqs = get_seqeval_preds(baseline_model, X_test, y_test)
print('=== Baseline BiLSTM Test Results ===')
print(classification_report(true_seqs, pred_seqs))

75/75 ━━━━━━━━━━━━━━━━━━━━ 9s 99ms/step
=== Baseline BiLSTM Test Results ===


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: IN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NNP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: , seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: DT seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarn

              precision    recall  f1-score   support

           B       0.97      0.95      0.96      4038
          BD       0.96      0.98      0.97      3832
          BG       0.95      0.96      0.95      1859
          BN       0.96      0.94      0.95      2955
          BP       0.98      0.97      0.97      1559
          BR       0.87      0.82      0.84       112
          BS       0.85      0.88      0.87        33
          BZ       0.99      0.97      0.98      2563
           C       1.00      1.00      1.00      2348
           D       1.00      0.97      0.98      2990
          DT       0.94      0.97      0.96       367
           H       0.67      1.00      0.80         2
           J       0.92      0.93      0.93      7028
          JR       0.94      0.94      0.94       289
          JS       0.96      0.97      0.96       290
           N       0.95      0.95      0.95     19512
          NP       0.89      0.91      0.90      9145
         NPS       0.83    

In [ ]:
def pos(sentence):
    sent = sent_tok.texts_to_sequences([sentence])
    sent_pad = pad_sequences(sent, maxlen=max_len, padding='post')
    pred = baseline_model.predict(sent_pad)
    prediction = np.argmax(pred, -1)[0]

    words = sentence.split()
    tags = [pos_tok.index_word.get(prediction[i], 'UNK') for i in range(len(words))]

    print(f'Sentence : {sentence}')
    print(f'POS      : {tags}')

pos('She is reading a book in the library .')
pos('The cat sat on the mat .')
pos("He quickly ran to catch the bus")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
Sentence : She is reading a book in the library .
POS      : ['PRP', 'VBZ', 'VBG', 'DT', 'NN', 'IN', 'DT', 'NN', '.']
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
Sentence : The cat sat on the mat .
POS      : ['DT', 'NN', 'VBD', 'IN', 'DT', 'NN', '.']
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
Sentence : He quickly ran to catch the bus
POS      : ['PRP', 'RB', 'VBD', 'TO', 'VB', 'DT', 'NN']


In [ ]:
import pickle
baseline_model.save('pos_model.keras')
pickle.dump(sent_tok, open('sent_token.pkl', 'wb'))
pickle.dump(pos_tok, open('pos_token.pkl', 'wb'))